# W8 — Variance Swap Hedging & VRP Backtest
Group 4 | Computational Finance

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import FuncFormatter
from scipy import stats

from src.models.heston import HestonParams, simulate_heston_paths
from src.pricing.variance_mc import compute_realized_variance, heston_variance_strike, vix_squared_from_variance
from src.pricing.vrp_backtest import (
    demonstrate_model_free_hedge, simulate_daily_pnl, run_vrp_backtest,
    HESTON_PARAMS, HESTON_PHYSICAL, N_MONTHS_BACKTEST, VEGA_TARGET, VIX_STOP_LOSS,
    plot_model_free_hedge, plot_daily_pnl, plot_backtest, print_summary,
)


## Part A — Model-Free Vega Hedge

Weights: $w(K) = \\frac{2 e^{rT} \\Delta K}{T K^2}$ — depend only on strike grid, not on model params.

In [ ]:
hedge_data = demonstrate_model_free_hedge()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# hedge weights curve
strikes = hedge_data['strikes']
weights = hedge_data['weights']
axes[0].plot(strikes, weights * 1e4, color='steelblue', lw=2.5)
axes[0].fill_between(strikes, weights * 1e4, alpha=0.2, color='steelblue')
axes[0].axvline(100, color='grey', ls='--', lw=1, label='ATM')
axes[0].set_title('w(K) = 2*exp(rT)*dK / (T*K^2)', fontsize=11)
axes[0].set_xlabel('Strike K')
axes[0].set_ylabel('Weight x 10^4')
axes[0].legend()
axes[0].grid(alpha=0.3)

# same weights, different IV surfaces
vals = [hedge_data['model_free_var_flat_surface']*10000,
        hedge_data['model_free_var_skewed_surface']*10000]
bars = axes[1].bar(['Flat IV (20%)', 'Skewed IV'], vals,
                   color=['steelblue', 'mediumpurple'], alpha=0.8, width=0.4)
for b, v in zip(bars, vals):
    axes[1].text(b.get_x()+b.get_width()/2, v+0.2, f'{v:.1f}', ha='center')
axes[1].set_title('Same weights -> different K_var (model-free)')
axes[1].set_ylabel('Implied Variance (x10^4)')
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../outputs/w8/w8_model_free_hedge.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Flat IV -> var = {vals[0]:.2f}  |  Skewed IV -> var = {vals[1]:.2f}")
print("Weights are identical in both cases -- model-free confirmed")


## Part B — Daily Vega P&L

In [ ]:
pnl_data = simulate_daily_pnl(HESTON_PARAMS, n_paths=500, seed=77)

days = range(len(pnl_data['mean_pnl_unhedged']))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(days, pnl_data['mean_pnl_unhedged'], color='tomato', lw=2, label='Unhedged')
axes[0].plot(days, pnl_data['mean_pnl_hedged'], color='seagreen', lw=2, label='Hedged')
axes[0].axhline(0, color='grey', ls='--', lw=0.8)
axes[0].set_title('Cumulative Vega P&L')
axes[0].set_xlabel('Trading Day')
axes[0].set_ylabel('P&L ($)')
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[0].yaxis.set_major_formatter(FuncFormatter(lambda x,_: f'${x:,.0f}'))

axes[1].hist(pnl_data['all_terminal_unhedged'], bins=35, color='tomato', alpha=0.6, density=True,
             label=f"Unhedged  sigma=${pnl_data['std_terminal_unhedged']:,.0f}")
axes[1].hist(pnl_data['all_terminal_hedged'], bins=35, color='seagreen', alpha=0.6, density=True,
             label=f"Hedged    sigma=${pnl_data['std_terminal_hedged']:,.0f}")
axes[1].set_title(f"Terminal P&L  |  Var reduction: {pnl_data['var_reduction_pct']:.1f}%")
axes[1].set_xlabel('Terminal P&L ($)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/w8/w8_daily_pnl.png', dpi=150, bbox_inches='tight')
plt.show()


## Part C — VRP Backtest 2019–2024

In [ ]:
result = run_vrp_backtest(
    params_rn=HESTON_PARAMS,
    params_p=HESTON_PHYSICAL,
    n_months=N_MONTHS_BACKTEST,
)

print_summary(
    demonstrate_model_free_hedge(),
    simulate_daily_pnl(HESTON_PARAMS, n_paths=500, seed=77),
    result,
)


In [ ]:
df  = result.monthly_pnl
eq  = result.equity_curve
m   = result.metrics
reg = result.regime_analysis

fig = plt.figure(figsize=(17, 12))
gs  = gridspec.GridSpec(3, 3, hspace=0.5, wspace=0.35)

# equity curve
ax1 = fig.add_subplot(gs[0, :2])
ax1.plot(eq.index, eq.values, color='steelblue', lw=2.5)
ax1.fill_between(eq.index, eq.values, 100, alpha=0.15, color='steelblue')
crisis = df[df['crisis']]
stress = df[df['stress']]
if not crisis.empty:
    ax1.axvspan(crisis.index[0], crisis.index[0]+pd.DateOffset(months=2),
                color='red', alpha=0.2, label='Crisis (Mar-2020)')
if not stress.empty:
    ax1.axvspan(stress.index[0], stress.index[0]+pd.DateOffset(months=2),
                color='orange', alpha=0.2, label='Stress (2022)')
stops = df[df['stop_loss']]
if not stops.empty:
    ax1.scatter(stops.index, eq.loc[stops.index], marker='x', color='orange', s=80, label='Stop-loss')
ax1.axhline(100, color='grey', ls='--', lw=0.7)
ax1.set_title('Equity Curve — Short Variance Swap (2019–2024)')
ax1.set_ylabel('NAV (base=100)')
ax1.legend(fontsize=8)
ax1.grid(alpha=0.25)

# monthly P&L bars
ax2 = fig.add_subplot(gs[1, :2])
bar_c = ['seagreen' if v > 0 else 'tomato' for v in df['net_pnl']]
ax2.bar(df.index, df['net_pnl'], color=bar_c, alpha=0.8, width=20)
ax2.axhline(0, color='grey', lw=0.8)
ax2.set_title('Monthly Net P&L ($)')
ax2.yaxis.set_major_formatter(FuncFormatter(lambda x,_: f'${x:,.0f}'))
ax2.grid(alpha=0.25)

# VIX level
ax3 = fig.add_subplot(gs[2, :2])
ax3.plot(df.index, df['vix_t'], color='mediumpurple', lw=1.8)
ax3.axhline(VIX_STOP_LOSS, color='orange', ls='--', lw=1.2, label=f'Stop-loss {VIX_STOP_LOSS}')
ax3.fill_between(df.index, df['vix_t'], alpha=0.1, color='mediumpurple')
ax3.set_title('Model VIX Level')
ax3.set_ylabel('VIX')
ax3.legend(fontsize=9)
ax3.grid(alpha=0.25)

# risk metrics table
ax4 = fig.add_subplot(gs[0, 2])
ax4.axis('off')
rows = [
    ('Ann. Return',  f"{m['annualised_return_pct']:.1f}%"),
    ('Ann. Vol',     f"{m['annualised_vol_pct']:.1f}%"),
    ('Sharpe',       f"{m['sharpe']:.2f}"),
    ('Sortino',      f"{m['sortino']:.2f}"),
    ('Max DD',       f"{m['max_drawdown_pct']:.1f}%"),
    ('Calmar',       f"{m['calmar']:.2f}"),
    ('VaR 5%',       f"{m['var_5pct']:.1f}%"),
    ('CVaR 5%',      f"{m['cvar_5pct']:.1f}%"),
    ('Skew',         f"{m['skewness']:.2f}"),
    ('Exc. Kurt',    f"{m['excess_kurtosis']:.2f}"),
    ('Win Rate',     f"{m['win_rate_pct']:.0f}%"),
]
ax4.text(0.5, 1.0, 'Risk Metrics', ha='center', fontsize=12, fontweight='bold',
         transform=ax4.transAxes)
for i, (lbl, val) in enumerate(rows):
    y = 0.92 - i*0.085
    ax4.text(0.05, y, lbl, transform=ax4.transAxes, fontsize=9, color='grey')
    ax4.text(0.98, y, val, transform=ax4.transAxes, fontsize=9, ha='right', fontweight='bold')

# return distribution
ax5 = fig.add_subplot(gs[1, 2])
r_norm = df['net_pnl'] / VEGA_TARGET
ax5.hist(r_norm, bins=20, color='steelblue', alpha=0.75, density=True, edgecolor='white')
xl = np.linspace(*ax5.get_xlim(), 200)
ax5.plot(xl, stats.norm.pdf(xl, r_norm.mean(), r_norm.std()), 'orange', lw=1.5, ls='--', label='Normal')
ax5.axvline(float(np.percentile(r_norm, 5)), color='tomato', lw=1.5, ls=':', label='VaR 5%')
ax5.set_title('Monthly Returns (negative skew expected)')
ax5.legend(fontsize=8)
ax5.grid(alpha=0.25)

# regime sharpe
ax6 = fig.add_subplot(gs[2, 2])
rlabels = ['High VRP\n(>3 vol)', 'Low VRP\n(<1 vol)', 'Full']
rs = [reg['high_vrp'].get('sharpe',0), reg['low_vrp'].get('sharpe',0), reg['full'].get('sharpe',0)]
bc = ['seagreen' if s > 0 else 'tomato' for s in rs]
bars = ax6.bar(rlabels, rs, color=bc, alpha=0.85, width=0.5)
ax6.axhline(0, color='grey', lw=0.8)
ax6.set_title('Sharpe by VRP Regime')
for bar, s in zip(bars, rs):
    ax6.text(bar.get_x()+bar.get_width()/2, s+0.03*(1 if s>=0 else -1),
             f'{s:.2f}', ha='center', fontsize=9, fontweight='bold')
ax6.grid(alpha=0.25, axis='y')

fig.suptitle('W8 — Short Variance Swap VRP Backtest (2019–2024)', fontsize=14, fontweight='bold')
plt.savefig('../outputs/w8/w8_backtest.png', dpi=150, bbox_inches='tight')
plt.show()


## Monthly P&L Table

In [ ]:
tbl = df[['vix_t','k_var','rv','vrp_vol','net_pnl','stop_loss','crisis']].copy()
tbl.columns = ['VIX','K_var','RV','VRP (vol pts)','Net P&L ($)','Stop-Loss','Crisis']
tbl['Net P&L ($)'] = tbl['Net P&L ($)'].map('${:,.0f}'.format)
tbl['VRP (vol pts)'] = tbl['VRP (vol pts)'].map('{:.2f}'.format)
tbl


## Factor Regression

$R^{VRP}_t = \\alpha + \\beta_1 R^{SPX}_t + \\beta_2 \\Delta VIX_t + \\beta_3 VIX_{t-1} + \\varepsilon_t$

In [ ]:
fr = result.factor_regression
print(f"alpha:       {fr['alpha_monthly']:.4f}")
print(f"beta_spx:    {fr['beta_spx']:.4f}")
print(f"beta_dvix:   {fr['beta_dvix']:.4f}  (negative -> VIX spike hurts)")
print(f"beta_vixlag: {fr['beta_vix_lag']:.4f}")
print(f"R2:          {fr['r_squared']:.3f}")
print()
print(f"Skewness = {m['skewness']:.2f}  -- small consistent gains, rare large losses")
